# 02 - Feature Engineering
**Objective:** Construct economically meaningful features from World Bank and WTO cleaned datasets.
**Features:**
1. Global Demand Index (WTO sector-level)
2. Demand Growth Rate (year-over-year)
3. Rolling 3-year average demand
4. Product Diversification Metrics (HHI, Shannon Entropy)
5. Economic Context: lagged GDP growth, trade volume proxy, import volume proxy, computed GDP per capita
**Input:** `data/processed/01_*.csv` | **Output:** `data/processed/02_*.csv`

In [1]:
import os, numpy as np, pandas as pd, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
PROJECT_ROOT = Path.cwd().parent.parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
df_wb = pd.read_csv(PROCESSED_DIR / '01_worldbank_cleaned.csv')
df_wto = pd.read_csv(PROCESSED_DIR / '01_wto_cleaned.csv')
print('World Bank shape:', df_wb.shape)
print('WTO shape:', df_wto.shape)

World Bank shape: (5375, 9)
WTO shape: (450, 10)


## 2.1 - WTO Demand Features

In [2]:
df_wto['year'] = pd.to_numeric(df_wto['year'], errors='coerce')
df_wto['value'] = pd.to_numeric(df_wto['value'], errors='coerce')
if 'indicator' in df_wto.columns:
    print('WTO indicators:')
    print(df_wto['indicator'].value_counts().head(10))
if 'product_sector' in df_wto.columns:
    wto_agg = df_wto.groupby(['year', 'product_sector'])['value'].sum().reset_index()
    yearly_max = wto_agg.groupby('year')['value'].transform('max')
    wto_agg['global_demand_index'] = wto_agg['value'] / yearly_max.replace(0, np.nan)
    wto_agg = wto_agg.sort_values(['product_sector', 'year']).copy()
    wto_agg['demand_growth_rate_pct'] = wto_agg.groupby('product_sector')['value'].pct_change() * 100
    wto_agg['demand_3yr_ma'] = wto_agg.groupby('product_sector')['value'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    print('WTO demand features computed. Shape:', wto_agg.shape)
    display(wto_agg.head(10))
else:
    wto_agg = pd.DataFrame()
    print('No product_sector column found in WTO data.')

WTO indicators:
indicator
Merchandise Imports By Product Group � Annual    181
Merchandise Exports By Product Group � Annual    181
Total Merchandise Imports - Quarterly             44
Total Merchandise Exports - Quarterly             44
Name: count, dtype: int64
WTO demand features computed. Shape: (181, 6)


,year,product_sector,value,global_demand_index,demand_growth_rate_pct,demand_3yr_ma
0,2015,Agricultural Products,11007.890104,0.063725,NaN,11007.890104
18,2016,Agricultural Products,9971.365942,0.064653,-9.416193,10489.628023
36,2017,Agricultural Products,10176.748441,0.062626,2.059723,10385.334829
54,2018,Agricultural Products,8674.170198,0.049214,-14.764817,9607.428194
72,2019,Agricultural Products,9145.108000,0.057193,5.429197,9332.008880
90,2020,Agricultural Products,9326.060000,0.081126,1.978675,9048.446066
108,2021,Agricultural Products,10786.671263,0.070872,15.661611,9752.613088
126,2022,Agricultural Products,12834.879324,0.061362,18.988324,10982.536862
144,2023,Agricultural Products,11793.402072,0.059928,-8.114430,11804.984220
162,2024,Agricultural Products,11924.018211,0.063156,1.107536,12184.099869


## 2.2 - Product Diversification Metrics (WTO)

In [3]:
if not df_wto.empty and 'product_sector' in df_wto.columns:
    wto_yearly = df_wto.groupby(['year', 'product_sector'])['value'].sum().reset_index()
    yearly_totals = wto_yearly.groupby('year')['value'].transform('sum')
    wto_yearly['share'] = wto_yearly['value'] / yearly_totals.replace(0, np.nan)
    hhi = wto_yearly.groupby('year').apply(lambda x: (x['share'] ** 2).sum()).reset_index(name='hhi')
    def shannon_entropy(shares):
        shares = shares[shares > 0]
        return -(shares * np.log(shares)).sum()
    entropy = wto_yearly.groupby('year').apply(lambda x: shannon_entropy(x['share'])).reset_index(name='shannon_entropy')
    diversification = hhi.merge(entropy, on='year')
    max_entropy = np.log(wto_yearly['product_sector'].nunique()) if wto_yearly['product_sector'].nunique() > 0 else 1
    diversification['shannon_entropy_norm'] = diversification['shannon_entropy'] / max_entropy
    print('Diversification metrics per year:')
    display(diversification)
else:
    diversification = pd.DataFrame()
    print('Skipping diversification.')

Diversification metrics per year:


,year,hhi,shannon_entropy,shannon_entropy_norm
0,2015,0.275791,1.811936,0.626887
1,2016,0.278157,1.806677,0.625067
2,2017,0.279159,1.792764,0.620254
3,2018,0.291695,1.733755,0.599838
4,2019,0.287931,1.764823,0.610587
5,2020,0.291371,1.775879,0.614412
6,2021,0.293458,1.733184,0.599640
7,2022,0.302313,1.643553,0.568630
8,2023,0.292247,1.710958,0.591951
9,2024,0.287524,1.741922,0.602664


## 2.3 - Economic Context Features (World Bank)

In [4]:
df_wb['year'] = pd.to_numeric(df_wb['year'], errors='coerce')
indicator_cols = ['NY.GDP.MKTP.CD', 'NY.GDP.MKTP.KD.ZG', 'NY.GDP.PCAP.CD', 'SP.POP.TOTL', 'TG.VAL.TOTL.GD.ZS', 'NE.IMP.GNFS.ZS']
for col in indicator_cols:
    if col in df_wb.columns:
        df_wb[col] = pd.to_numeric(df_wb[col], errors='coerce')
growth_col = 'NY.GDP.MKTP.KD.ZG'
if growth_col in df_wb.columns:
    df_wb = df_wb.sort_values(['country_code', 'year']).copy()
    df_wb['gdp_growth_lag1'] = df_wb.groupby('country_code')[growth_col].shift(1)
trade_col = 'TG.VAL.TOTL.GD.ZS'
gdp_col = 'NY.GDP.MKTP.CD'
if trade_col in df_wb.columns and gdp_col in df_wb.columns:
    df_wb['trade_volume_proxy'] = df_wb[trade_col] * df_wb[gdp_col] / 100
imp_col = 'NE.IMP.GNFS.ZS'
if imp_col in df_wb.columns and gdp_col in df_wb.columns:
    df_wb['import_volume_proxy'] = df_wb[imp_col] * df_wb[gdp_col] / 100
pop_col = 'SP.POP.TOTL'
if gdp_col in df_wb.columns and pop_col in df_wb.columns:
    df_wb['gdp_per_capita_computed'] = df_wb[gdp_col] / df_wb[pop_col].replace(0, np.nan)
print('Economic context features added. Sample:')
display(df_wb.head())
print('New columns:', [c for c in df_wb.columns if c not in indicator_cols + ['country', 'country_code', 'year']])

Economic context features added. Sample:


,country,country_code,year,NE.IMP.GNFS.ZS,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,NY.GDP.PCAP.CD,SP.POP.TOTL,TG.VAL.TOTL.GD.ZS,gdp_growth_lag1,trade_volume_proxy,import_volume_proxy,gdp_per_capita_computed
200,Aruba,ABW,2000.0,70.686869,1.873453e+09,7.622921,20681.023027,90588.0,272.544938,NaN,5.106000e+09,1.324285e+09,20681.023027
201,Aruba,ABW,2001.0,69.394325,1.896457e+09,4.182002,20740.132583,91439.0,252.839903,7.622921,4.795000e+09,1.316034e+09,20740.132583
202,Aruba,ABW,2002.0,68.666458,1.961844e+09,-0.944953,21307.248251,92074.0,178.658484,4.182002,3.505000e+09,1.347128e+09,21307.248251
203,Aruba,ABW,2003.0,70.063078,2.044112e+09,1.110505,21949.485996,93128.0,217.649551,-0.944953,4.449000e+09,1.432168e+09,21949.485996
204,Aruba,ABW,2004.0,67.765371,2.254831e+09,7.293728,23700.631990,95138.0,311.508971,1.110505,7.024000e+09,1.527994e+09,23700.631990


New columns: ['gdp_growth_lag1', 'trade_volume_proxy', 'import_volume_proxy', 'gdp_per_capita_computed']


## 2.4 - Save Feature-Engineered Datasets

In [5]:
if not wto_agg.empty:
    wto_agg.to_csv(PROCESSED_DIR / '02_wto_demand_features.csv', index=False)
    print('Saved: 02_wto_demand_features.csv')
if not diversification.empty:
    diversification.to_csv(PROCESSED_DIR / '02_diversification_metrics.csv', index=False)
    print('Saved: 02_diversification_metrics.csv')
df_wb.to_csv(PROCESSED_DIR / '02_worldbank_features.csv', index=False)
print('Saved: 02_worldbank_features.csv')
print('\nFeature engineering complete. Proceed to 03_scaling_and_normalization.ipynb.')

Saved: 02_wto_demand_features.csv
Saved: 02_diversification_metrics.csv


Saved: 02_worldbank_features.csv

Feature engineering complete. Proceed to 03_scaling_and_normalization.ipynb.
